In [1]:
#Import dependencies:

from ipyleaflet import (Map, basemaps, GeoJSON, FullScreenControl, WidgetControl)
from ipywidgets import HTML, Button

import geopandas as gpd
import pandas as pd
import json

In [2]:
#Geospatial Data Loading

#Boundary data of PFAs
with open("PFA_(2021)_BGC.geojson",'r') as pfa:
    pfa_data = json.load(pfa)
    
#Boundary data of MSOAs
with open("MSOA_(2021)_BGC.geojson",'r') as msoa:
    msoa_data = json.load(msoa)

In [3]:
#Mapping Data:

#MSOA to LAD mapping table
MSOA_to_LAD = pd.read_csv("OA_to_LSOA_to_MSOA_to_LAD_(December 2021).csv", usecols=["MSOA21CD", "MSOA21NM", "LAD22CD", "LAD22NM"]).drop_duplicates().rename(columns={"LAD22CD":"LAD21CD", "LAD22NM":"LAD21NM"})

#LAD to PFA mapping table
LAD_to_PFA = pd.read_excel("LAD_to_PFA_(December 2021).xlsx", usecols=["LAD21CD", "LAD21NM", "PFA21CD", "PFA21NM"]).drop_duplicates()

#Merging the previous two mapping tables
MSOA_to_PFA = pd.merge(MSOA_to_LAD, LAD_to_PFA, on="LAD21CD", how="inner").drop(["LAD21CD", "LAD21NM_x", "LAD21NM_y"], axis=1)

In [4]:
#Creating the interactive visualisation

center = [53,0]
zoom = 5.5
m = Map(basemap=basemaps.CartoDB.Positron, center=center, zoom=zoom)

pfa_layer = GeoJSON(data=pfa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.03, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)
msoa_layer = GeoJSON(data=msoa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.03, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)



html_pfa = HTML('''<h3><b>Hover over a Police Force!</b></h3>''')
html_pfa.layout.margin = '0px 20px 20px 20px'
pfa_control = WidgetControl(widget=html_pfa, position='topright')

html_msoa = HTML('''<h3><b>Hover over an MSOA!</b></h3>''')
html_msoa.layout.margin = '0px 20px 20px 20px'
msoa_control = WidgetControl(widget=html_msoa, position='topright')

def update_pfa_info(**kwargs):
    html_pfa.value = '''
                 <h3><b>Police Force: </b>{}</h3>
                 <h4>PFA Code: {}</h4>
                 '''.format(kwargs['properties']['PFA21NM'], kwargs['properties']['PFA21CD'])
    
def update_msoa_info(**kwargs):
    html_msoa.value = '''
                 <h3><b>MSOA: </b>{}</h3>
                 <h4>MSOA Code: {}</h4>
                 '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'])



return_to_pfa_button = Button(
    description="Return to PFA view",
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip="Return to PFA view",
    icon="arrow-left"
)
return_control = WidgetControl(widget=return_to_pfa_button, position='topright')
def back_to_pfa(button_instance):
    m.substitute(msoa_layer, pfa_layer)
    m.substitute(msoa_control, pfa_control)
    m.remove(return_control)
    m.center =  center
    m.zoom = zoom



def whenClicked_PFA(**kwargs):
    MSOAs_in_PFA = MSOA_to_PFA.loc[MSOA_to_PFA["PFA21CD"] == kwargs["properties"]["PFA21CD"]]["MSOA21CD"]
    MSOAs_json = {
                  "type": "FeatureCollection", 
                  "crs": {"type": "name", "properties": {"name": "EPSG:4326"}},
                  "features": []
                 }
    for i in msoa_data["features"]:
        for j in MSOAs_in_PFA:
            if i["properties"]["MSOA21CD"] == j:
                MSOAs_json["features"].append(i)
                break
    msoa_layer.data = MSOAs_json
    m.substitute(pfa_layer, msoa_layer)
    m.substitute(pfa_control, msoa_control)
    m.add(return_control)
    m.center =  [kwargs['properties']['LAT'], kwargs['properties']['LONG']]
    m.zoom = 9



pfa_layer.on_click(whenClicked_PFA)
pfa_layer.on_hover(update_pfa_info)

msoa_layer.on_hover(update_msoa_info)

return_to_pfa_button.on_click(back_to_pfa)

m.add(pfa_layer)
m.add(FullScreenControl())
m.add(pfa_control)

m

Map(center=[53, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…